In [ ]:
!pip install -q gradio transformers torch torchvision torchaudio soundfile

In [ ]:
import gradio as gr
from transformers import pipeline
import warnings

# Ignorar avisos de depreciação de bibliotecas no Colab para manter o log limpo
warnings.filterwarnings("ignore")

print("Carregando modelos base... (Isso pode levar alguns minutos)")

# 1. CARREGAMENTO DOS MODELOS BASE
sentiment_pipe = pipeline("sentiment-analysis")
text_gen_pipe = pipeline("text-generation", model="gpt2")
zero_shot_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
ner_pipe = pipeline("ner", aggregation_strategy="simple")
tts_pipe = pipeline("text-to-speech", model="kakao-enterprise/vits-ljs")

# 2. CARREGAMENTO DOS NOVOS MODELOS (O DESAFIO)
print("Carregando novos modelos de IA...")

# Nova Funcionalidade 1: Classificação de Imagens (ViT)
image_class_pipe = pipeline("image-classification", model="google/vit-base-patch16-224")

# Nova Funcionalidade 2: Speech-to-Text (Whisper da OpenAI)
asr_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")

# Nova Funcionalidade 3: Image-to-Text (BLIP da Salesforce - Legenda de Imagens)
image_caption_pipe = pipeline("image-text-to-text", model="Salesforce/blip-image-captioning-base")

print("Todos os modelos foram carregados com sucesso!")

# FUNÇÕES DE PROCESSAMENTO

def analyze_sentiment(text):
    if not text: return "Por favor, insira um texto."
    result = sentiment_pipe(text)[0]
    return f"Sentimento: {result['label']} (Confiança: {result['score']:.2f})"

def generate_text(prompt):
    if not prompt: return "Por favor, insira um prompt."
    result = text_gen_pipe(prompt, max_length=50, num_return_sequences=1)[0]
    return result['generated_text']

def zero_shot_classify(text, candidate_labels):
    if not text or not candidate_labels: return "Preencha o texto e as categorias."
    labels = [label.strip() for label in candidate_labels.split(",")]
    result = zero_shot_pipe(text, labels)
    output = "Resultados da Classificação:\n"
    for label, score in zip(result['labels'], result['scores']):
        output += f"- {label}: {score:.1%}\n"
    return output

def extract_entities(text):
    if not text: return "Por favor, insira um texto."
    result = ner_pipe(text)
    if not result: return "Nenhuma entidade encontrada."
    output = "Entidades Encontradas:\n"
    for entity in result:
        output += f"[{entity['entity_group']}] {entity['word']} (Confiança: {entity['score']:.2f})\n"
    return output

def text_to_speech(text):
    if not text: return None
    result = tts_pipe(text)
    return (result["sampling_rate"], result["audio"][0].numpy())

def classify_image(image):
    if image is None: return "Por favor, envie uma imagem."
    result = image_class_pipe(image)
    output = "Previsões do Modelo:\n"
    for res in result:
        output += f"- {res['label']} ({res['score']:.2%})\n"
    return output

def speech_to_text(audio_path):
    if not audio_path: return "Por favor, grave ou envie um áudio."
    result = asr_pipe(audio_path)
    return result["text"]

def caption_image(image):
    if image is None: return "Por favor, envie uma imagem."
    try:
        # Passar text="" satisfaz a exigência do novo pipeline unificado
        result = image_caption_pipe(image, text="")
        return result[0]["generated_text"]
    except Exception as e:
        # Se ocorrer falta de memória (OOM) ou outro erro, este aparecerá na caixa de texto do Gradio
        return f"Erro interno do modelo: {str(e)}"

# INTERFACE GRADIO

with gr.Blocks(theme=gr.themes.Soft()) as super_app:
    gr.Markdown(
        """
        # Super App GenAI
        **Projeto Integrador de IA** | Aplicativo rodando múltiplos modelos Open-Source localmente via Hugging Face Transformers.
        """
    )

    with gr.Tabs():

        # ABA 1: Funcionalidades Base
        with gr.Tab("Funcionalidades Base"):
            with gr.Accordion("🎭 Análise de Sentimentos", open=False):
                with gr.Row():
                    sent_in = gr.Textbox(label="Digite um texto (em inglês)")
                    sent_out = gr.Textbox(label="Resultado")
                gr.Button("Analisar").click(analyze_sentiment, inputs=sent_in, outputs=sent_out)

            with gr.Accordion("✍️ Geração de Texto", open=False):
                with gr.Row():
                    gen_in = gr.Textbox(label="Prompt inicial (em inglês)")
                    gen_out = gr.Textbox(label="Texto Gerado")
                gr.Button("Gerar").click(generate_text, inputs=gen_in, outputs=gen_out)

            with gr.Accordion("🏷️ Classificação Zero-Shot", open=False):
                with gr.Row():
                    zs_in_text = gr.Textbox(label="Texto (em inglês)")
                    zs_in_labels = gr.Textbox(label="Categorias (separadas por vírgula)")
                zs_out = gr.Textbox(label="Classificação")
                gr.Button("Classificar").click(zero_shot_classify, inputs=[zs_in_text, zs_in_labels], outputs=zs_out)

            with gr.Accordion("🔍 Extração de Entidades (NER)", open=False):
                with gr.Row():
                    ner_in = gr.Textbox(label="Texto (em inglês)")
                    ner_out = gr.Textbox(label="Entidades Encontradas")
                gr.Button("Extrair").click(extract_entities, inputs=ner_in, outputs=ner_out)

            with gr.Accordion("🗣️ Text-to-Speech (TTS)", open=False):
                with gr.Row():
                    tts_in = gr.Textbox(label="Texto para áudio (em inglês)")
                    tts_out = gr.Audio(label="Áudio Gerado", type="numpy")
                gr.Button("Gerar Áudio").click(text_to_speech, inputs=tts_in, outputs=tts_out)

        # ABA 2: Novas Funcionalidades
        with gr.Tab("Novas Funcionalidades de IA"):

            with gr.Accordion("🖼️ 1. Classificação de Imagens", open=True):
                gr.Markdown("Identifica o objeto principal presente em uma imagem usando Vision Transformer (ViT).")
                with gr.Row():
                    img_in = gr.Image(type="pil", label="Faça o upload de uma imagem")
                    img_out = gr.Textbox(label="Resultados da Classificação")
                gr.Button("Classificar Imagem", variant="primary").click(classify_image, inputs=img_in, outputs=img_out)

            with gr.Accordion("🎙️ 2. Speech-to-Text (Transcrição)", open=False):
                gr.Markdown("Transforma áudio gravado em texto usando o modelo Whisper.")
                with gr.Row():
                    # Usa filepath para melhor compatibilidade com o Colab
                    asr_in = gr.Audio(type="filepath", label="Grave ou faça upload de um áudio")
                    asr_out = gr.Textbox(label="Texto Transcrito")
                gr.Button("Transcrever", variant="primary").click(speech_to_text, inputs=asr_in, outputs=asr_out)

            with gr.Accordion("📝 3. Image-to-Text (Legenda de Imagens)", open=False):
                gr.Markdown("Lê uma imagem e escreve uma legenda descrevendo o que está acontecendo nela.")
                with gr.Row():
                    cap_in = gr.Image(type="pil", label="Faça o upload de uma imagem")
                    cap_out = gr.Textbox(label="Legenda Gerada (em inglês)")
                gr.Button("Gerar Legenda", variant="primary").click(caption_image, inputs=cap_in, outputs=cap_out)

if __name__ == "__main__":
    super_app.launch(share=True, debug=True)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Carregando modelos base... (Isso pode levar alguns minutos)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Todos os modelos foram carregados com sucesso!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9609cdd299cab659a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
